In [ ]:
%pip install -U langgraph


  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.19
    Uninstalling langgraph-1.2.19:
      Successfully uninstalled langgraph-1.2.19
Note: you may need to restart the kernel to use updated packages.


In [47]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
import re

In [48]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

api_key = getpass("Cole sua OpenAI API Key: ").strip()

os.environ["OPENAI_API_KEY"] = api_key

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=api_key
)

In [49]:
class State(TypedDict):
    """ Representa o estado do processo de avaliação da redação """
    essay: str
    relevance_score: float
    grammar_score: float
    structure_score: float
    depth_score: float
    final_score: float

In [50]:
#Funções de Avaliação
#Esta célula define as funções utilizadas no processo de avaliação, incluindo a extração de pontuações e os componentes individuais de avaliação.

In [51]:
def extract_score(content: str) -> float:
    """Extrai a pontuação numérica da resposta do LLM."""
    match = re.search(r'Pontuação:\s*(\d+(\.\d+)?)', content)
    if match:
        return float(match.group(1))
    raise ValueError(f"Não foi possível extrair a pontuação de: {content}")

def check_relevance(state: State) -> State:
    """Verifica a relevância da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a relevância da seguinte redação em relação ao tema dado, prezando pela excelência da Língua Portuguesa. "
        "Forneça uma pontuação de relevância entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["relevance_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_relevance: {e}")
        state["relevance_score"] = 0.0
    return state

def check_grammar(state: State) -> State:
    """Verifica a gramática da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a gramática da Língua Portuguesa na seguinte redação. "
        "Forneça uma pontuação de gramática entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["grammar_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_grammar: {e}")
        state["grammar_score"] = 0.0
    return state

def analyze_structure(state: State) -> State:
    """Analisa a estrutura da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a estrutura de acordo com a normal culta da Língua Portuguesa na seguinte redação. "
        "Forneça uma pontuação de estrutura entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["structure_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em analyze_structure: {e}")
        state["structure_score"] = 0.0
    return state

def evaluate_depth(state: State) -> State:
    """Avalia a profundidade de análise na redação."""
    prompt = ChatPromptTemplate.from_template(
        "Avalie a profundidade de análise na seguinte redação. "
        "Forneça uma pontuação de profundidade entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["depth_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em evaluate_depth: {e}")
        state["depth_score"] = 0.0
    return state

def calculate_final_score(state: State) -> State:
    """Calcula a pontuação final com base nas pontuações dos componentes individuais."""
    state["final_score"] = (
        state["relevance_score"] * 0.3 +
        state["grammar_score"] * 0.2 +
        state["structure_score"] * 0.2 +
        state["depth_score"] * 0.3
    )
    return state

In [52]:
#Definição do fluxo de trabalho
#Essa célula define o fluxo de trabalho de avaliação usando StateGraph.

In [74]:
# Inicializa o StateGraph
workflow = StateGraph(State)

# Adiciona nós ao grafo
workflow.add_node("check_relevance", check_relevance)
workflow.add_node("check_grammar", check_grammar)
workflow.add_node("analyze_structure", analyze_structure)
workflow.add_node("evaluate_depth", evaluate_depth)
workflow.add_node("calculate_final_score", calculate_final_score)

# Define as funções de decisão

# def decidir_relevancia(state):
#     if state["relevance_score"] > 0.5:
#         return "check_grammar"
#     else:
#         return "calculate_final_score"


# def decidir_gramatica(state):
#     if state["grammar_score"] > 0.6:
#         return "analyze_structure"
#     else:
#         return "calculate_final_score"


# def decidir_estrutura(state):
#     if state["structure_score"] > 0.7:
#         return "evaluate_depth"
#     else:
#         return "calculate_final_score"


# # Adiciona as arestas condicionais

# workflow.add_conditional_edges(
#     "check_relevance",
#     decidir_relevancia
# )

# workflow.add_conditional_edges(
#     "check_grammar",
#     decidir_gramatica
# )

# workflow.add_conditional_edges(
#     "analyze_structure",
#     decidir_estrutura
# )

# # Aresta sem condição
# workflow.add_edge(
#     "evaluate_depth",
#     "calculate_final_score"
# )




# Define e adiciona arestas condicionais
workflow.add_conditional_edges(
    "check_relevance",
    lambda x: (
        "check_grammar"
        if x["relevance_score"] > 0.5
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "check_grammar",
    lambda x: (
        "analyze_structure"
        if x["relevance_score"] > 0.6
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "analyze_structure",
    lambda x: (
        "evaluate_depth"
        if x["relevance_score"] > 0.7
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "evaluate_depth",
    lambda x: (
    "calculate_final_score"
    )
)


    
# # Define e adiciona arestas condicionais
# workflow.add_conditional_edges(
#     "check_relevance",
#     lambda x: "check_grammar" if x["relevance_score"] > 0.5 else "calculate_final_score"
    
# )
# workflow.add_conditional_edges(
#     "check_grammar",
#     lambda x: "analyze_structure" if x["grammar_score"] > 0.6 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "analyze_structure",
#     lambda x: "evaluate_depth" if x["structure_score"] > 0.7 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "evaluate_depth",
#     lambda x: "calculate_final_score"
# )

# Define o ponto de entrada
workflow.set_entry_point("check_relevance")

# Define o ponto de saída
workflow.add_edge("calculate_final_score", END)


# Compila o grafo
app = workflow.compile()

In [75]:
# workflow.add_conditional_edges(
#     "check_relevance",
#     decidir_proximo_no
# )
# workflow.add_conditional_edges(
#     "check_grammar",
#     decidir_proximo_no
# )
# workflow.add_conditional_edges(
#     "check_relevance",
#     decidir_proximo_no
# )
# # Define e adiciona arestas condicionais
# workflow.add_conditional_edges(
#     "check_relevance",
#     lambda x: "check_grammar" if x["relevance_score"] > 0.5 else "calculate_final_score"
    
# )
# workflow.add_conditional_edges(
#     "check_grammar",
#     lambda x: "analyze_structure" if x["grammar_score"] > 0.6 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "analyze_structure",
#     lambda x: "evaluate_depth" if x["structure_score"] > 0.7 else "calculate_final_score"
# )
# workflow.add_conditional_edges(
#     "evaluate_depth",
#     lambda x: "calculate_final_score"
# )

# #Função de avaliação de redação
# #Essa célula define a função principal para avaliar uma redação usando o fluxo de trabalho definido.

In [76]:
def grade_essay(essay: str) -> dict:
    """Avalia a redação fornecida usando o fluxo de trabalho definido."""
    initial_state = State(
        essay=essay,
        relevance_score=0.0,
        grammar_score=0.0,
        structure_score=0.0,
        depth_score=0.0,
        final_score=0.0
    )
    result = app.invoke(initial_state)
    return result

In [77]:
sample_essay = """
O Impacto da Computação Quântica na Inteligência Artificial

A evolução tecnológica tem transformado profundamente a sociedade, especialmente com o avanço da Inteligência Artificial (IA) e o desenvolvimento da computação quântica. Enquanto a inteligência artificial permite que máquinas aprendam padrões, analisem informações e auxiliem na tomada de decisões, a computação quântica propõe uma nova forma de processamento baseada nos princípios da mecânica quântica. A integração dessas duas tecnologias representa uma possibilidade de avanço científico, capaz de transformar diferentes setores da economia e da sociedade.

Um dos principais impactos da computação quântica na inteligência artificial está relacionado à possibilidade de acelerar determinados cálculos computacionais. Diferentemente dos computadores tradicionais, que utilizam bits representados pelos valores 0 ou 1, os computadores quânticos trabalham com qubits, que podem apresentar estados de superposição. Essa característica, combinada com outros fenômenos quânticos, permite desenvolver algoritmos com vantagens computacionais para problemas específicos. Na inteligência artificial, essa capacidade poderá contribuir para determinadas tarefas de otimização, análise de dados e aprendizado de máquina, embora ainda existam desafios para demonstrar vantagens práticas em larga escala.

Além disso, a combinação entre essas tecnologias poderá beneficiar áreas como medicina, indústria e pesquisa científica. Na saúde, por exemplo, algoritmos de inteligência artificial associados a métodos quânticos poderão auxiliar na descoberta de medicamentos e no estudo de estruturas moleculares. No setor industrial, poderão contribuir para a otimização de processos produtivos, planejamento logístico e desenvolvimento de novos materiais. Entretanto, muitas dessas aplicações ainda estão em fase experimental, sendo necessário aprimorar tanto os equipamentos quânticos quanto os algoritmos utilizados.

Apesar das perspectivas positivas, a integração da computação quântica com a inteligência artificial também apresenta desafios importantes. Os computadores quânticos atuais são sensíveis a ruídos e erros, além de exigirem condições especiais de funcionamento. Outro aspecto relevante envolve questões econômicas e sociais, pois o elevado custo dessa tecnologia poderá concentrar seu desenvolvimento em grandes empresas e instituições de pesquisa. Dessa forma, investimentos em educação, formação profissional e acesso à pesquisa serão fundamentais para ampliar os benefícios dessas inovações.

Portanto, a computação quântica possui potencial para influenciar significativamente o futuro da inteligência artificial, oferecendo novas possibilidades para a resolução de problemas complexos. Contudo, seu impacto dependerá da superação de limitações técnicas, da comprovação de vantagens computacionais e da democratização do acesso a essas tecnologias. Assim, a colaboração entre universidades, governos e empresas será essencial para garantir que esses avanços contribuam de maneira responsável para o desenvolvimento científico, econômico e social.
    """

In [78]:
#Avaliação da redação de exemplo
#Essa célula demonstra como usar o sistema de avaliação na redação de exemplo e exibir os resultados.
#mudou de 0-1 pra 0-10

In [79]:
# Avalia a redação de exemplo

result = grade_essay(sample_essay)

# Converte as pontuações de 0-1 para 0-10
final_score = result['final_score'] * 10
relevance_score = result['relevance_score'] * 10
grammar_score = result['grammar_score'] * 10
structure_score = result['structure_score'] * 10
depth_score = result['depth_score'] * 10

# Exibe os resultados
print(f"Pontuação Final da Redação: {final_score:.2f}/10\n")
print(f"Pontuação de Relevância: {relevance_score:.2f}/10")
print(f"Pontuação de Gramática: {grammar_score:.2f}/10")
print(f"Pontuação de Estrutura: {structure_score:.2f}/10")
print(f"Pontuação de Profundidade: {depth_score:.2f}/10")

Pontuação Final da Redação: 8.97/10

Pontuação de Relevância: 9.70/10
Pontuação de Gramática: 9.80/10
Pontuação de Estrutura: 9.40/10
Pontuação de Profundidade: 7.40/10
